<a href="https://colab.research.google.com/github/Dasheeer/lab_REST-API-React/blob/main/REST_API_React_%D0%A1%D1%82%D1%83%D0%B4%D0%B5%D0%BD%D1%87%D0%B5%D1%81%D0%BA%D0%B8%D0%B9_%D0%B6%D1%83%D1%80%D0%BD%D0%B0%D0%BB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip install fastapi uvicorn sqlalchemy nest_asyncio pyngrok

In [1]:

import time
import threading
import nest_asyncio
import uvicorn

from fastapi import FastAPI, HTTPException, Path, Query
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field, ConfigDict
from sqlalchemy import create_engine, Column, Integer, String, ForeignKey
from sqlalchemy.orm import declarative_base, sessionmaker
from pyngrok import ngrok

# Разрешаем запуск сервера в Google Colab
nest_asyncio.apply()

# 1. Создаём приложение
app = FastAPI(
    title="API студенческого журнала",
    version="1.0.0"
)
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=False,
    allow_methods=["*"],
    allow_headers=["*"],
)
# 2. Подключаем базу данных SQLite
DATABASE_URL = "sqlite:///students.db"

engine = create_engine(
    DATABASE_URL,
    connect_args={"check_same_thread": False}
)

SessionLocal = sessionmaker(
    autocommit=False,
    autoflush=False,
    bind=engine
)

Base = declarative_base()


# 3. Модель студента
class Student(Base):
    __tablename__ = "students"

    id = Column(Integer, primary_key=True, index=True)
    name = Column(String, nullable=False)
    group_name = Column(String, nullable=False)


# 4. Модель оценки
class Grade(Base):
    __tablename__ = "grades"

    id = Column(Integer, primary_key=True, index=True)
    student_id = Column(
        Integer,
        ForeignKey("students.id"),
        nullable=False
    )
    subject = Column(String, nullable=False)
    grade = Column(Integer, nullable=False)


# Создаём таблицы, если их ещё нет
Base.metadata.create_all(bind=engine)


# 5. Схема данных студента
class StudentCreate(BaseModel):
    name: str = Field(
        min_length=1,
        description="Фамилия и имя студента",
        examples=["Анна Смирнова"]
    )
    group_name: str = Field(
        min_length=1,
        description="Название учебной группы",
        examples=["БСТ2556"]
    )

    model_config = ConfigDict(
        json_schema_extra={
            "example": {
                "name": "Анна Смирнова",
                "group_name": "БСТ2556"
            }
        }
    )


# 6. Схема данных оценки
class GradeCreate(BaseModel):
    subject: str = Field(
        min_length=1,
        description="Название учебной дисциплины",
        examples=["Базы данных"]
    )
    grade: int = Field(
        ge=2,
        le=5,
        description="Оценка по пятибалльной шкале: от 2 до 5",
        examples=[5]
    )

    model_config = ConfigDict(
        json_schema_extra={
            "example": {
                "subject": "Базы данных",
                "grade": 5
            }
        }
    )


# 7. Преобразование данных в JSON
def student_to_dict(student):
    return {
        "id": student.id,
        "name": student.name,
        "group_name": student.group_name
    }


def grade_to_dict(grade):
    return {
        "id": grade.id,
        "student_id": grade.student_id,
        "subject": grade.subject,
        "grade": grade.grade
    }


# 8. Получить список всех студентов — GET
@app.get(
    "/students",
    tags=["Студенты"],
    summary="Получить список студентов",
    description=(
        "Возвращает список студентов. "
        "Можно фильтровать по учебной группе "
        "и сортировать по имени."
    )
)
def get_students(
    group_name: str | None = Query(
        default=None,
        description="Фильтр по названию учебной группы",
        examples=["БСТ2556"]
    ),
    sort_order: str = Query(
        default="asc",
        pattern="^(asc|desc)$",
        description=(
            "Порядок сортировки по имени: "
            "asc — по алфавиту, desc — в обратном порядке"
        )
    )
):
    db = SessionLocal()
    try:
        query = db.query(Student)

        # Фильтрация по учебной группе
        if group_name:
            query = query.filter(
                Student.group_name == group_name
            )

        # Сортировка по имени
        if sort_order == "asc":
            query = query.order_by(Student.name.asc())
        else:
            query = query.order_by(Student.name.desc())

        students = query.all()

        return [student_to_dict(s) for s in students]
    finally:
        db.close()



# 9. Получить информацию о студенте — GET
@app.get(
    "/students/{student_id}",
    tags=["Студенты"],
    summary="Получить информацию о студенте",
    description="Возвращает данные студента по его идентификатору."
)
def get_student(
    student_id: int = Path(
        ...,
        description="Уникальный идентификатор студента"
    )
):
    db = SessionLocal()
    try:
        student = db.query(Student).filter(
            Student.id == student_id
        ).first()

        if student is None:
            raise HTTPException(
                status_code=404,
                detail="Студент не найден"
            )

        return student_to_dict(student)
    finally:
        db.close()


# 10. Добавить студента — POST
@app.post(
    "/students",
    tags=["Студенты"],
    status_code=201,
    summary="Добавить студента",
    description="Создаёт новую запись о студенте."
)
def create_student(data: StudentCreate):
    db = SessionLocal()
    try:
        student = Student(
            name=data.name,
            group_name=data.group_name
        )
        db.add(student)
        db.commit()
        db.refresh(student)

        return student_to_dict(student)
    finally:
        db.close()


# 11. Изменить данные студента — PUT
@app.put(
    "/students/{student_id}",
    tags=["Студенты"],
    summary="Изменить данные студента",
    description="Обновляет имя и учебную группу студента."
)
def update_student(student_id: int, data: StudentCreate):
    db = SessionLocal()
    try:
        student = db.query(Student).filter(
            Student.id == student_id
        ).first()

        if student is None:
            raise HTTPException(
                status_code=404,
                detail="Студент не найден"
            )

        student.name = data.name
        student.group_name = data.group_name

        db.commit()
        db.refresh(student)

        return student_to_dict(student)
    finally:
        db.close()


# 12. Удалить студента — DELETE
@app.delete(
    "/students/{student_id}",
    tags=["Студенты"],
    summary="Удалить студента",
    description="Удаляет студента и все его оценки."
)
def delete_student(student_id: int):
    db = SessionLocal()
    try:
        student = db.query(Student).filter(
            Student.id == student_id
        ).first()

        if student is None:
            raise HTTPException(
                status_code=404,
                detail="Студент не найден"
            )

        db.query(Grade).filter(
            Grade.student_id == student_id
        ).delete()

        db.delete(student)
        db.commit()

        return {"message": "Студент успешно удалён"}
    finally:
        db.close()


# 13. Добавить оценку студенту — POST
@app.post(
    "/students/{student_id}/grades",
    tags=["Студенты"],
    status_code=201,
    summary="Добавить оценку студенту",
    description="Добавляет студенту оценку по дисциплине."
)
def create_grade(student_id: int, data: GradeCreate):
    db = SessionLocal()
    try:
        student = db.query(Student).filter(
            Student.id == student_id
        ).first()

        if student is None:
            raise HTTPException(
                status_code=404,
                detail="Студент не найден"
            )

        grade = Grade(
            student_id=student_id,
            subject=data.subject,
            grade=data.grade
        )

        db.add(grade)
        db.commit()
        db.refresh(grade)

        return grade_to_dict(grade)
    finally:
        db.close()


# 14. Получить оценки студента — GET
@app.get(
    "/students/{student_id}/grades",
    tags=["Студенты"],
    summary="Получить оценки студента",
    description="Возвращает все оценки студента по дисциплинам."
)
def get_grades(student_id: int):
    db = SessionLocal()
    try:
        student = db.query(Student).filter(
            Student.id == student_id
        ).first()

        if student is None:
            raise HTTPException(
                status_code=404,
                detail="Студент не найден"
            )

        grades = db.query(Grade).filter(
            Grade.student_id == student_id
        ).all()

        return [grade_to_dict(g) for g in grades]
    finally:
        db.close()


# 15. Изменить оценку — PUT
@app.put(
    "/grades/{grade_id}",
    tags=["Студенты"],
    summary="Изменить оценку",
    description="Изменяет дисциплину или балл существующей оценки."
)
def update_grade(grade_id: int, data: GradeCreate):
    db = SessionLocal()
    try:
        grade = db.query(Grade).filter(
            Grade.id == grade_id
        ).first()

        if grade is None:
            raise HTTPException(
                status_code=404,
                detail="Оценка не найдена"
            )

        grade.subject = data.subject
        grade.grade = data.grade

        db.commit()
        db.refresh(grade)

        return grade_to_dict(grade)
    finally:
        db.close()


# 16. Удалить оценку — DELETE
@app.delete(
    "/grades/{grade_id}",
    tags=["Студенты"],
    summary="Удалить оценку",
    description="Удаляет выбранную оценку из журнала."
)
def delete_grade(grade_id: int):
    db = SessionLocal()
    try:
        grade = db.query(Grade).filter(
            Grade.id == grade_id
        ).first()

        if grade is None:
            raise HTTPException(
                status_code=404,
                detail="Оценка не найдена"
            )

        db.delete(grade)
        db.commit()

        return {"message": "Оценка успешно удалена"}
    finally:
        db.close()


# 17. Запуск сервера
server = uvicorn.Server(
    uvicorn.Config(
        app,
        host="0.0.0.0",
        port=8002,
        log_level="warning"
    )
)

api_thread = threading.Thread(
    target=server.run,
    daemon=True
)
api_thread.start()

# Проверяем, запустился ли сервер
for _ in range(30):
    if server.started:
        break
    time.sleep(0.5)

if not server.started:
    raise RuntimeError(
        "Сервер не запустился. Возможно, порт 8002 занят. "
        "Перезапусти среду выполнения Colab и попробуй снова."
    )

# 18. Создаём публичную ссылку для Swagger
ngrok.kill()
tunnel = ngrok.connect(8002)
public_url = tunnel.public_url

print("API успешно запущен!")
print("Ссылка на API:", public_url)
print("Документация Swagger:", public_url + "/docs")
print("База данных SQLite готова к работе.")



API успешно запущен!
Ссылка на API: https://e1f1-136-109-219-79.ngrok-free.app
Документация Swagger: https://e1f1-136-109-219-79.ngrok-free.app/docs
База данных SQLite готова к работе.


In [4]:
import pyngrok
print("pyngrok установлен")

pyngrok установлен


In [3]:
pip install pyngrok

In [3]:
from fastapi.responses import HTMLResponse

html = r"""
<!DOCTYPE html>
<html lang="ru">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1">
  <title>Студенческий журнал</title>

  <script src="https://unpkg.com/react@18/umd/react.development.js"></script>
  <script src="https://unpkg.com/react-dom@18/umd/react-dom.development.js"></script>
  <script src="https://unpkg.com/@babel/standalone/babel.min.js"></script>

  <style>
    * { box-sizing: border-box; }
    body {
      font-family: Arial, sans-serif;
      background: #f3f5f9;
      margin: 0;
      padding: 24px;
      color: #20283a;
    }
    main {
      max-width: 1050px;
      margin: auto;
      background: white;
      padding: 26px;
      border-radius: 14px;
      box-shadow: 0 4px 20px #1c2b4a0b;
    }
    h1 { margin-top: 0; }
    h2, h3 { margin-top: 22px; }
    .subtitle { color: #667085; }
    nav {
      display: flex;
      gap: 8px;
      flex-wrap: wrap;
      margin: 22px 0;
      border-bottom: 1px solid #e4e7ee;
      padding-bottom: 16px;
    }
    input, select, button {
      padding: 10px 12px;
      margin: 5px 5px 5px 0;
      border: 1px solid #ccd2df;
      border-radius: 7px;
      font-size: 14px;
    }
    input, select { max-width: 100%; }
    button {
      cursor: pointer;
      background: #2457d6;
      color: white;
      border: 0;
    }
    button:hover { opacity: .9; }
    button.active { background: #173b96; }
    button.delete { background: #c0392b; }
    button.secondary { background: #65718a; }
    button.light { background: #e9efff; color: #2457d6; }
    .panel {
      background: #f7f8fb;
      padding: 16px;
      border-radius: 10px;
      margin: 14px 0;
    }
    .stats {
      display: flex;
      gap: 12px;
      flex-wrap: wrap;
      margin: 16px 0;
    }
    .stat {
      flex: 1;
      min-width: 150px;
      padding: 16px;
      background: #f1f5ff;
      border-radius: 10px;
    }
    .stat strong { display: block; font-size: 25px; margin-top: 6px; }
    table {
      border-collapse: collapse;
      width: 100%;
      margin-top: 16px;
    }
    th, td {
      text-align: left;
      border-bottom: 1px solid #e4e7ee;
      padding: 12px 8px;
      vertical-align: top;
    }
    th { background: #f7f8fb; }
    .table-wrap { width: 100%; overflow-x: auto; }
    .message {
      padding: 12px;
      background: #fff0ef;
      color: #b42318;
      border-radius: 8px;
      margin: 14px 0;
      overflow-wrap: anywhere;
    }
    .success {
      padding: 12px;
      background: #ecfdf3;
      color: #067647;
      border-radius: 8px;
      margin: 14px 0;
    }
    .form-grid {
      display: grid;
      grid-template-columns: repeat(2, minmax(0, 1fr));
      gap: 12px;
    }
    .field { display: flex; flex-direction: column; }
    .field label { font-size: 14px; font-weight: bold; margin-bottom: 4px; }
    .field input { width: 100%; margin: 0; }
    .actions { display: flex; flex-wrap: wrap; gap: 4px; }
    .detail-row { padding: 12px 0; border-bottom: 1px solid #e4e7ee; }
    .detail-label { color: #667085; display: block; font-size: 13px; margin-bottom: 4px; }
    .empty { color: #667085; padding: 18px 0; }
    @media (max-width: 650px) {
      body { padding: 10px; }
      main { padding: 16px; }
      .form-grid { grid-template-columns: 1fr; }
      th, td { padding: 9px 6px; }
    }
  </style>
</head>

<body>
  <main>
    <div id="root">Загрузка приложения...</div>
  </main>

  <script type="text/babel">
    const { useState, useEffect } = React;

    function App() {
      const [screen, setScreen] = useState("list");
      const [students, setStudents] = useState([]);
      const [search, setSearch] = useState("");
      const [group, setGroup] = useState("Все группы");
      const [form, setForm] = useState(null);
      const [details, setDetails] = useState(null);
      const [grades, setGrades] = useState([]);
      const [error, setError] = useState("");
      const [success, setSuccess] = useState("");
      const [loading, setLoading] = useState(false);

      async function request(path, options = {}) {
        const response = await fetch(path, {
          ...options,
          headers: {
            "Content-Type": "application/json",
            "ngrok-skip-browser-warning": "true",
            ...(options.headers || {})
          }
        });

        if (!response.ok) {
          const detail = await response.text();
          throw new Error("Ошибка " + response.status + ": " + detail);
        }

        if (response.status === 204) return null;
        return response.json();
      }

      async function loadStudents() {
        setLoading(true);
        setError("");
        try {
          const data = await request("/students");
          setStudents(data);
        } catch (e) {
          setError("Не удалось загрузить студентов: " + e.message);
        } finally {
          setLoading(false);
        }
      }

      useEffect(() => {
        loadStudents();
      }, []);

      const groups = [
        ...new Set(students.map(s => s.group_name).filter(Boolean))
      ];

      const filtered = students.filter(s =>
        (s.name || "").toLowerCase().includes(search.toLowerCase()) &&
        (group === "Все группы" || s.group_name === group)
      );

      function openCreateForm() {
        setError("");
        setSuccess("");
        setForm({ id: null, name: "", group_name: "" });
        setScreen("form");
      }

      function openEditForm(student) {
        setError("");
        setSuccess("");
        setForm({
          id: student.id,
          name: student.name,
          group_name: student.group_name
        });
        setScreen("form");
      }

      async function saveStudent(e) {
        e.preventDefault();
        setError("");
        setSuccess("");

        const payload = {
          name: form.name.trim(),
          group_name: form.group_name.trim()
        };

        if (!payload.name || !payload.group_name) {
          setError("Заполни имя студента и учебную группу.");
          return;
        }

        try {
          if (form.id !== null) {
            await request("/students/" + form.id, {
              method: "PUT",
              body: JSON.stringify(payload)
            });
            setSuccess("Данные студента успешно изменены.");
          } else {
            await request("/students", {
              method: "POST",
              body: JSON.stringify(payload)
            });
            setSuccess("Студент успешно добавлен.");
          }

          setForm(null);
          await loadStudents();
          setScreen("list");
        } catch (e) {
          setError("Не удалось сохранить запись: " + e.message);
        }
      }

      async function deleteStudent(id) {
        if (!confirm("Удалить студента?")) return;
        setError("");
        setSuccess("");

        try {
          await request("/students/" + id, { method: "DELETE" });
          setSuccess("Студент успешно удалён.");
          if (details && details.id === id) {
            setDetails(null);
            setGrades([]);
          }
          await loadStudents();
        } catch (e) {
          setError("Не удалось удалить студента: " + e.message);
        }
      }

      async function openDetails(studentId) {
        setError("");
        setSuccess("");
        setDetails(null);
        setGrades([]);
        setScreen("details");

        try {
          const student = await request("/students/" + studentId);
          setDetails(student);

          try {
            const data = await request("/students/" + studentId + "/grades");
            setGrades(data);
          } catch (e) {
            setError("Данные студента загружены, но оценки получить не удалось: " + e.message);
          }
        } catch (e) {
          setError("Не удалось загрузить информацию о студенте: " + e.message);
        }
      }

      function goToScreen(name) {
        setError("");
        setSuccess("");
        setScreen(name);
      }

      return (
        <div>
          <h1>Студенческий журнал</h1>
          <p className="subtitle">
            Управление студентами и оценками с помощью React и REST API
          </p>

          <nav>
            <button
              className={screen === "list" ? "active" : "secondary"}
              onClick={() => goToScreen("list")}
            >
              1. Список студентов
            </button>
            <button
              className={screen === "form" ? "active" : "secondary"}
              onClick={openCreateForm}
            >
              2. Добавить студента
            </button>
            <button
              className={screen === "details" ? "active" : "secondary"}
              onClick={() => goToScreen("details")}
            >
              3. Поиск и детали
            </button>
          </nav>

          {error && <div className="message">{error}</div>}
          {success && <div className="success">{success}</div>}

          {screen === "list" && (
            <section>
              <h2>Реестр студентов</h2>
              <p className="subtitle">
                Просмотр записей, редактирование и удаление студентов.
              </p>

              <div className="stats">
                <div className="stat">
                  Всего студентов
                  <strong>{students.length}</strong>
                </div>
                <div className="stat">
                  Учебных групп
                  <strong>{groups.length}</strong>
                </div>
              </div>

              <button onClick={openCreateForm}>+ Добавить студента</button>
              <button className="secondary" onClick={loadStudents}>
                Обновить список
              </button>

              {loading ? <p>Загрузка списка...</p> : (
                <div className="table-wrap">
                  <table>
                    <thead>
                      <tr>
                        <th>ID</th>
                        <th>Имя</th>
                        <th>Группа</th>
                        <th>Действия</th>
                      </tr>
                    </thead>
                    <tbody>
                      {students.map(s => (
                        <tr key={s.id}>
                          <td>{s.id}</td>
                          <td>{s.name}</td>
                          <td>{s.group_name}</td>
                          <td>
                            <div className="actions">
                              <button className="light"
                                onClick={() => openDetails(s.id)}>
                                Подробнее
                              </button>
                              <button onClick={() => openEditForm(s)}>
                                Изменить
                              </button>
                              <button className="delete"
                                onClick={() => deleteStudent(s.id)}>
                                Удалить
                              </button>
                            </div>
                          </td>
                        </tr>
                      ))}
                      {students.length === 0 && (
                        <tr>
                          <td colSpan="4" className="empty">
                            Список студентов пуст.
                          </td>
                        </tr>
                      )}
                    </tbody>
                  </table>
                </div>
              )}
            </section>
          )}

          {screen === "form" && (
            <section>
              <h2>
                {form && form.id !== null
                  ? "Редактирование студента"
                  : "Добавление студента"}
              </h2>
              <p className="subtitle">
                Заполни данные и сохрани их в базе данных через REST API.
              </p>

              {form && (
                <form className="panel" onSubmit={saveStudent}>
                  <div className="form-grid">
                    <div className="field">
                      <label htmlFor="student-name">Имя студента</label>
                      <input
                        id="student-name"
                        required
                        minLength="1"
                        maxLength="100"
                        placeholder="Например, Анна Смирнова"
                        value={form.name}
                        onChange={e =>
                          setForm({ ...form, name: e.target.value })
                        }
                      />
                    </div>

                    <div className="field">
                      <label htmlFor="student-group">Учебная группа</label>
                      <input
                        id="student-group"
                        required
                        minLength="1"
                        maxLength="50"
                        placeholder="Например, БСТ2556"
                        value={form.group_name}
                        onChange={e =>
                          setForm({ ...form, group_name: e.target.value })
                        }
                      />
                    </div>
                  </div>

                  <div style={{ marginTop: "16px" }}>
                    <button type="submit">Сохранить</button>
                    <button
                      type="button"
                      className="secondary"
                      onClick={() => {
                        setForm(null);
                        setError("");
                        setScreen("list");
                      }}
                    >
                      Отмена
                    </button>
                  </div>
                </form>
              )}
            </section>
          )}

          {screen === "details" && (
            <section>
              <h2>Поиск, фильтрация и подробная информация</h2>
              <p className="subtitle">
                Найди студента по имени или группе, затем открой его карточку.
              </p>

              <div className="panel">
                <div className="form-grid">
                  <div className="field">
                    <label htmlFor="search-name">Поиск по имени</label>
                    <input
                      id="search-name"
                      placeholder="Введи имя или его часть"
                      value={search}
                      onChange={e => setSearch(e.target.value)}
                    />
                  </div>

                  <div className="field">
                    <label htmlFor="filter-group">Фильтр по группе</label>
                    <select
                      id="filter-group"
                      value={group}
                      onChange={e => setGroup(e.target.value)}
                    >
                      <option>Все группы</option>
                      {groups.map(g => <option key={g} value={g}>{g}</option>)}
                    </select>
                  </div>
                </div>

                <p>Найдено студентов: <strong>{filtered.length}</strong></p>

                <div className="table-wrap">
                  <table>
                    <thead>
                      <tr>
                        <th>ID</th>
                        <th>Имя</th>
                        <th>Группа</th>
                        <th>Карточка</th>
                      </tr>
                    </thead>
                    <tbody>
                      {filtered.map(s => (
                        <tr key={s.id}>
                          <td>{s.id}</td>
                          <td>{s.name}</td>
                          <td>{s.group_name}</td>
                          <td>
                            <button className="light"
                              onClick={() => openDetails(s.id)}>
                              Открыть
                            </button>
                          </td>
                        </tr>
                      ))}
                      {filtered.length === 0 && (
                        <tr>
                          <td colSpan="4" className="empty">
                            По заданным условиям ничего не найдено.
                          </td>
                        </tr>
                      )}
                    </tbody>
                  </table>
                </div>
              </div>

              {details && (
                <div className="panel">
                  <h3>Карточка студента</h3>

                  <div className="detail-row">
                    <span className="detail-label">ID</span>
                    <strong>{details.id}</strong>
                  </div>
                  <div className="detail-row">
                    <span className="detail-label">Имя</span>
                    <strong>{details.name}</strong>
                  </div>
                  <div className="detail-row">
                    <span className="detail-label">Учебная группа</span>
                    <strong>{details.group_name}</strong>
                  </div>

                  <h3>Оценки по дисциплинам</h3>
                  {grades.length > 0 ? (
                    <div className="table-wrap">
                      <table>
                        <thead>
                          <tr>
                            <th>ID оценки</th>
                            <th>Дисциплина</th>
                            <th>Оценка</th>
                          </tr>
                        </thead>
                        <tbody>
                          {grades.map(g => (
                            <tr key={g.id}>
                              <td>{g.id}</td>
                              <td>{g.subject}</td>
                              <td>{g.grade}</td>
                            </tr>
                          ))}
                        </tbody>
                      </table>
                    </div>
                  ) : (
                    <p className="empty">
                      Оценок пока нет или API не вернул данные.
                    </p>
                  )}

                  <button onClick={() => openEditForm(details)}>
                    Редактировать студента
                  </button>
                </div>
              )}
            </section>
          )}
        </div>
      );
    }

    ReactDOM.createRoot(document.getElementById("root")).render(<App />);
  </script>
</body>
</html>
"""

@app.get("/", response_class=HTMLResponse)
def student_journal_page():
    return HTMLResponse(content=html)